# Artist Collaboration Network — Data Preparation

Cleans and merges raw data (nodes, edges, previews, images) into enriched CSVs for the scrollytelling visualization.

## 1. Load raw data

In [1]:
import pandas as pd
import numpy as np

edges = pd.read_csv('../data/edges.csv')
nodes = pd.read_csv('../data/nodes.csv')
images = pd.read_csv('../data/images.csv')
previews_raw = pd.read_csv('../data/spotify-previews.csv')

previews = previews_raw.loc[
    previews_raw['artists_coincides'] & previews_raw['track_name_conincides']
].copy()

print(f"{len(nodes)} nodes/artists")
print(f"{len(images)} artist images")
print(f"{len(edges)} edges/collaborations")
print(f"{len(previews)} preview URLs")

864 nodes/artists
864 artist images
8808 edges/collaborations
8224 preview URLs


## 2. Build indexed lookups

In [2]:
previews_idx = previews.set_index(['source', 'target'])[['spotify_artists', 'preview', 'id']]
nodes_idx = nodes.set_index('id')
images_idx = images.set_index('id')

## 3. Build enriched edges

In [3]:
edges_df = (
    edges[['source', 'target', 'weight', 'collab_track_name']]
    .rename(columns={'collab_track_name': 'track_name'})
    .merge(
        previews[['source', 'target', 'spotify_artists', 'preview', 'id']]
            .rename(columns={'spotify_artists': 'artists', 'id': 'track_id'}),
        on=['source', 'target'],
        how='left'
    )
    .merge(
        nodes[['id', 'id.1']].rename(columns={'id': 'source', 'id.1': 'source_id'}),
        on='source',
        how='left'
    )
    .merge(
        nodes[['id', 'id.1']].rename(columns={'id': 'target', 'id.1': 'target_id'}),
        on='target',
        how='left'
    )
)

edges_df['artists'] = edges_df['artists'].fillna('')
edges_df['preview'] = edges_df['preview'].fillna('')
edges_df['track_id'] = edges_df['track_id'].fillna('')

edges_df.head()

,source,target,weight,track_name,artists,preview,track_id,source_id,target_id
0,Bad Bunny,Jhay Cortez,6,Tarot,"Bad Bunny, Jhay Cortez",https://p.scdn.co/mp3-preview/db8db5f15768f239...,41oY4WCTj5kccfesTVFnvN,4q3ewBCX7sLwd24euuV69X,0EFisYRi20PTADoJrifHrz
1,Bad Bunny,Tony Dize,1,La Corriente,"Bad Bunny, Tony Dize",https://p.scdn.co/mp3-preview/5b652748ee9a5a10...,1797zYiX4cKosMH836X9Gt,4q3ewBCX7sLwd24euuV69X,3LKXWvXFWrkwUzJWxzwVpW
2,Bad Bunny,Rauw Alejandro,1,Party,"Bad Bunny, Rauw Alejandro",https://p.scdn.co/mp3-preview/4982ab22ba534ab7...,4tYFy8ALRjIZvnvSLw5lxN,4q3ewBCX7sLwd24euuV69X,1mcTU81TzQhprhouKaTkpq
3,Bad Bunny,Bomba Estéreo,1,Ojitos Lindos,"Bad Bunny, Bomba Estéreo",https://p.scdn.co/mp3-preview/3cd0d6fb21cdca04...,3k3NWokhRRkEPhCzPmV8TW,4q3ewBCX7sLwd24euuV69X,5n9bMYfz9qss2VOW89EVs2
4,Bad Bunny,Buscabulla,1,Andrea,"Bad Bunny, Buscabulla",https://p.scdn.co/mp3-preview/09536841505c5860...,44XjoNvtwevktFKjvVe1vH,4q3ewBCX7sLwd24euuV69X,0MoaBi6dSquXp6rrlqlF8R


## 4. Build enriched nodes

In [4]:
nodes_df = (
    nodes[['id.1', 'id', 'label', 'followers', 'popularity',
           'betweenesscentrality', 'modularity_class',
           'unique_collabs', 'total_collabs']]
    .rename(columns={
        'id.1': 'spotify_id',
        'id': 'name_id',
        'label': 'name',
        'betweenesscentrality': 'betweeness_centrality',
        'modularity_class': 'group',
    })
    .merge(images.rename(columns={'id': 'spotify_id', 'image_url': 'image'}),
           on='spotify_id', how='left')
)

nodes_df.head()

,spotify_id,name_id,name,followers,popularity,betweeness_centrality,group,unique_collabs,total_collabs,image
0,4q3ewBCX7sLwd24euuV69X,Bad Bunny,Bad Bunny,52915846,100,16172.977050,0,119,245,https://i.scdn.co/image/ab6761610000e5eb8ee9a6...
1,0EFisYRi20PTADoJrifHrz,Jhay Cortez,Jhay Cortez,4404989,84,2403.215884,0,82,188,https://i.scdn.co/image/ab6761610000e5ebbbeb7f...
2,3LKXWvXFWrkwUzJWxzwVpW,Tony Dize,Tony Dize,2991179,75,1241.701113,3,25,68,https://i.scdn.co/image/0fb75163d5836f4479ddc9...
3,1mcTU81TzQhprhouKaTkpq,Rauw Alejandro,Rauw Alejandro,12993677,89,13977.908156,0,148,286,https://i.scdn.co/image/ab6761610000e5eb75a694...
4,5n9bMYfz9qss2VOW89EVs2,Bomba Estéreo,Bomba Estéreo,973375,79,2390.747683,6,6,7,https://i.scdn.co/image/ab6761610000e5ebcc4d7c...


## 5. Exploration

In [5]:
print("betweeness_centrality:")
print("  min", nodes_df['betweeness_centrality'].min())
print("  max", nodes_df['betweeness_centrality'].max())

betweeness_centrality:
  min 0.0
  max 29520.020794


In [6]:
bizarrap_edges = edges_df[
    (edges_df['source'] == 'Bizarrap') | (edges_df['target'] == 'Bizarrap')
]
print(f"Bizarrap as source: {len(edges_df[edges_df['source'] == 'Bizarrap'])}")
print(f"Bizarrap as target: {len(edges_df[edges_df['target'] == 'Bizarrap'])}")
bizarrap_edges.head()

Bizarrap as source: 21
Bizarrap as target: 34


,source,target,weight,track_name,artists,preview,track_id,source_id,target_id
434,Nicky Jam,Bizarrap,1,"Nicky Jam: Bzrp Music Sessions, Vol. 41","Bizarrap, Nicky Jam",https://p.scdn.co/mp3-preview/fc6ec26be9a4ab64...,03LfOYi0icz4souspZVVhq,1SupJlEpv7RS2tPNRaHViT,716NhGYqD1jl2wI1Qkgq36
706,Anuel AA,Bizarrap,1,"Anuel AA: Bzrp Music Sessions, Vol. 46","Bizarrap, Anuel AA",https://p.scdn.co/mp3-preview/fe6c981136a44bbf...,1mrGEef8GuaEnDdw8J5BQp,2R21vXR83lH98kGeO99Y66,716NhGYqD1jl2wI1Qkgq36
895,Duki,Bizarrap,4,Malbec,"Duki, Bizarrap",https://p.scdn.co/mp3-preview/3a8a070801aff6e5...,6KEb17S00Inf0v1qYDgUAj,1bAftSH8umNcGZ0uyV7LMg,716NhGYqD1jl2wI1Qkgq36
1125,Eladio Carrion,Bizarrap,1,Sin Frenos,"Eladio Carrion, Bizarrap, Duki",https://p.scdn.co/mp3-preview/fc300b5690ffc3a3...,1b62AO1IzcVr5SOgoguc9o,5XJDexmWFLWOkjOEjOVX3e,716NhGYqD1jl2wI1Qkgq36
1784,Cazzu,Bizarrap,1,"Cazzu: Bzrp Music Sessions, Vol. 32","Bizarrap, Cazzu",https://p.scdn.co/mp3-preview/865e07feb536cef9...,1MZqHpFyB4j5tFCFNvURou,6w3SkAHYPsQ1bxV7VDlG5y,716NhGYqD1jl2wI1Qkgq36


In [7]:
edges_df['track_name'].value_counts().head(20)

track_name
Sin Ropa - Remix                                                                              17
Babymama - Remix                                                                              15
Se Le Ve                                                                                      15
Teca (Remix) [feat. Neo Pistea, Midel, Rei & Zecca]                                           15
Quizas                                                                                        14
Ella - Remix                                                                                  14
Travesuras - Remix                                                                            14
Bonita - Remix                                                                                14
Mi Cuarto 2                                                                                   14
Ven A Cantar                                                                                  12
Relación - Remix   

## 6. Top artists by followers and popularity

In [8]:
top_followers = (
    nodes_df[['name', 'followers']]
    .sort_values('followers', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_followers.index += 1
top_followers['followers'] = top_followers['followers'].map('{:,.0f}'.format)
top_followers

,name,followers
1,Bad Bunny,"52,915,846"
2,J Balvin,"34,781,550"
3,Ozuna,"32,957,080"
4,Maluma,"31,778,738"
5,Daddy Yankee,"28,854,310"
6,KAROL G,"27,055,406"
7,Shakira,"24,393,175"
8,Anuel AA,"23,045,207"
9,Sebastian Yatra,"19,625,821"
10,Paulo Londra,"17,795,379"


In [9]:
top_popularity = (
    nodes_df[['name', 'popularity']]
    .sort_values('popularity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top_popularity.index += 1
top_popularity

,name,popularity
1,Bad Bunny,100
2,Rauw Alejandro,89
3,J Balvin,88
4,Bizarrap,87
5,Daddy Yankee,87
6,KAROL G,86
7,Feid,85
8,Anuel AA,85
9,Shakira,85
10,Ozuna,85


## 7. Scrollytelling subset

In [10]:
SCROLLY_ARTISTS = ['Shakira', 'Rauw Alejandro', 'Bad Bunny', 'Nicky Jam']

scrolly_nodes = nodes_df[nodes_df['name_id'].isin(SCROLLY_ARTISTS)]
scrolly_edges = edges_df[
    edges_df['source'].isin(SCROLLY_ARTISTS) &
    edges_df['target'].isin(SCROLLY_ARTISTS)
]

print(f"{len(scrolly_nodes)} artists, {len(scrolly_edges)} edges")
scrolly_edges

4 artists, 6 edges


,source,target,weight,track_name,artists,preview,track_id,source_id,target_id
2,Bad Bunny,Rauw Alejandro,1,Party,"Bad Bunny, Rauw Alejandro",https://p.scdn.co/mp3-preview/4982ab22ba534ab7...,4tYFy8ALRjIZvnvSLw5lxN,4q3ewBCX7sLwd24euuV69X,1mcTU81TzQhprhouKaTkpq
10,Bad Bunny,Nicky Jam,4,Te Boté - Remix,"Nio Garcia, Casper Magico, Bad Bunny, Darell, ...",https://p.scdn.co/mp3-preview/8dc93bba37297078...,3V8UKqhEK5zBkBb6d6ub8i,4q3ewBCX7sLwd24euuV69X,1SupJlEpv7RS2tPNRaHViT
161,Rauw Alejandro,Shakira,1,Te Felicito,"Shakira, Rauw Alejandro",https://p.scdn.co/mp3-preview/8e39a11448120d03...,2rurDawMfoKP4uHyb2kJBt,1mcTU81TzQhprhouKaTkpq,0EmeFodog0BfCgMzAIvKQp
407,Nicky Jam,Rauw Alejandro,4,Que Le De,"Rauw Alejandro, Nicky Jam",https://p.scdn.co/mp3-preview/74068faefe53d881...,08aYFNUTIOMGq93e2VSArQ,1SupJlEpv7RS2tPNRaHViT,1mcTU81TzQhprhouKaTkpq
3652,Shakira,Nicky Jam,2,Perro Fiel (feat. Nicky Jam),"Shakira, Nicky Jam",https://p.scdn.co/mp3-preview/46f72efb73767025...,70lnL3QaSOIIyMa2X9aVRL,0EmeFodog0BfCgMzAIvKQp,1SupJlEpv7RS2tPNRaHViT
3662,Shakira,Rauw Alejandro,1,Te Felicito,"Shakira, Rauw Alejandro",https://p.scdn.co/mp3-preview/8e39a11448120d03...,2rurDawMfoKP4uHyb2kJBt,0EmeFodog0BfCgMzAIvKQp,1mcTU81TzQhprhouKaTkpq


## 8. Write output

In [11]:
nodes_df.to_csv('../output/nodes-full.csv', index=False)
edges_df.to_csv('../output/edges-full.csv', index=False)
scrolly_nodes.to_csv('../output/scrolly-artists.csv', index=False)
scrolly_edges.to_csv('../output/scrolly-edges.csv', index=False)

print("Done.")

Done.
